# 03 · Watchers, evaluators & launching a full experiment

Once `.env` / `braintrust.env` are configured, this notebook shows the complete
experiment lifecycle used by the production scripts:

1. **Preflight** — validate prompt + dataset with zero model credits.
2. **Evaluators** — the three Braintrust scorers registered by the runner.
3. **Launch** — start a full experiment run.
4. **Watchers** — monitor progress from the JSONL manifest; resume after a crash.
5. **Report** — score locally, then generate summary/report/charts.


## 0. Bootstrap: repo path + credentials

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "constants.py").exists()
)
sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)


In [ ]:
from src.braintrust_config import load_braintrust_config
from src.env_utils import require_env

config = load_braintrust_config()      # braintrust.env first, then .env
api_key = require_env("OPENROUTER_API_KEY")[0]

print("project:", config.project_name)
print("project_id:", config.project_id)
print("dataset:", config.dataset_project, "/", config.dataset)
print("model:", config.model)
print("braintrust api_key set:", bool(config.api_key))
print("openrouter api_key set:", bool(api_key))


## 1. Preflight (zero credits)

`preflight_eval.py` checks that the prompt version resolves and the dataset is
reachable under the current credentials **without sending any model request**.
Run it before any eval to catch setup problems early.

In [ ]:
import subprocess
import sys


def run_script(rel_script: str, *args: str) -> None:
    cmd = [sys.executable, str(ROOT / "scripts" / rel_script), *args]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)


run_script("braintrust/preflight_eval.py", "--dataset", config.dataset, "--prompt-version", "v17.2")


## 2. Evaluators registered by the runner

The eval runner wraps the OpenAI client with `braintrust.wrap_openai()` and runs
`braintrust.Eval(..., scores=[...])`. Exactly three scorers are registered:

- **`exact_match`** — `output.strip().lower() == expected_class`, scored 1.0/0.0.
- **`failure`** — rows whose output starts with `ERROR: ` (count as misses too).
- **`cost`** — each row's actual billed USD from OpenRouter's `usage.cost`.

Near-miss (runner-up == expected while predicted != expected) is **not** a Braintrust
scorer — it is computed locally from the runner-up line the manifest records, by
`score_manifest.py`.

Abridged registration from `braintrust_openrouter_input.py`:

```python
from braintrust import Eval, wrap_openai
from openai import OpenAI

client = wrap_openai(OpenAI(base_url=OPENROUTER_BASE_URL, api_key=key))

Eval(
    dataset=dataset_rows,
    task=classify_row,          # returns (output, metadata) per row
    scores=[exact_match, failure, cost],
    metadata={"model": model, "prompt_version": prompt_version},
    max_concurrency=8,
)
```


## 3. Launch a full experiment

Launch the runner in the background against the configured dataset. Each completed row
is written to a **local JSONL manifest** (the durable checkpoint) as well as Braintrust,
so the run survives crashes, Braintrust limits, and quota errors.

In [ ]:
experiment_name = f"notebook_full_{config.model.replace('/', '_')}_v17.2"
manifest = ROOT / "reports" / "manifests" / f"{experiment_name}.jsonl"

cmd = [
    sys.executable,
    str(ROOT / "scripts" / "braintrust" / "braintrust_openrouter_input.py"),
    "--dataset", config.dataset,
    "--prompt-version", "v17.2",
    "--model", config.model,
    "--experiment-name", experiment_name,
    "--manifest", str(manifest),
]
print("$", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=ROOT)
print("Launched PID:", proc.pid)


## 4. Watch the run from the manifest

Re-run this cell as the eval runs: it counts the final status per unique filename.
Each manifest record carries a `status` (`completed` / `error` / `empty`) plus
`runner_up` and `cost`; the eval runner retries transient provider failures up to
`MAX_TRIES=3`, growing `max_tokens` toward `MAX_TOKENS_CAP=32768` on length caps.

In [ ]:
import json
from pathlib import Path


def manifest_status_counts(path: Path) -> dict:
    counts: dict[str, int] = {}
    if not path.exists():
        return {"(manifest not created yet)": 0}
    for line in path.read_text().splitlines()[1:]:
        if not line.strip():
            continue
        rec = json.loads(line)
        status = rec.get("status", "empty")
        counts[status] = counts.get(status, 0) + 1
    return counts


manifest_status_counts(manifest)


## 5. Crash-proof resume

If a run dies (crash, Braintrust cap, quota 403), re-invoke the runner through
`resume_until_complete.py` until `--expected-rows` unique filenames have a final
status. Completed rows are skipped; failed/error rows are re-attempted. On completion
it auto-scores the manifest locally with `score_manifest.py` (no Braintrust scorer
credits). For production, `run_eval_queue.py` chains multiple jobs sequentially with
preflight checks and manifest verification between jobs.

In [ ]:
# Illustrative: re-invokes the runner until every row is finished.
# Expected rows must equal the dataset slice size (fixed_size_sampled = 160).
cmd = [
    sys.executable,
    str(ROOT / "scripts" / "braintrust" / "resume_until_complete.py"),
    "--dataset", config.dataset,
    "--prompt-version", "v17.2",
    "--model", config.model,
    "--max-tokens", "8192",
    "--experiment-name", experiment_name,
    "--manifest", str(manifest),
    "--expected-rows", "160",
]
print("$", " ".join(cmd))
# subprocess.run(cmd, cwd=ROOT, check=True)   # uncomment to run


## 6. Post-run scoring & reporting

The full reporting chain (also wired in `scripts/braintrust/`):

1. `score_manifest.py` — local scoring from the manifest, no Braintrust credits.
2. `summarize_braintrust_experiment.py` — per-image OK/MISS summary + exact_match.
3. `braintrust_report.py` — accuracy, confusion matrix (PNG+MD), misclassification
   reasoning, cost breakdown (adjust `--input-price`/`--output-price` to the current
   OpenRouter model rates).
4. `braintrust_metrics_visual.py` — per-class chart + heatmap, and appends the
   experiment to `docs/experiments/experiment_log.md`.

In [ ]:
run_script("braintrust/score_manifest.py", "--manifest", str(manifest))

run_script("braintrust/summarize_braintrust_experiment.py", "--experiment", experiment_name)

run_script(
    "braintrust/braintrust_report.py",
    "--experiment", experiment_name,
    "--model", config.model,
    "--prompt-version", "v17.2",
    "--dataset", config.dataset,
    "--images-per-class", "10",
    "--input-price", "0.03",
    "--output-price", "0.13",
)

run_script("braintrust/braintrust_metrics_visual.py", experiment_name)


## Recap

1. Preflight validates prompt + dataset with zero credits.
2. The runner registers `exact_match`, `failure`, and `cost` scorers.
3. A full run writes every row to the local manifest as well as Braintrust.
4. Watch progress from the manifest; resume with `resume_until_complete.py`.
5. Score locally, then generate the summary / report / charts / experiment log.

Inspect individual row traces in the Braintrust UI (each span carries `raw_response`,
`reasoning`, `model`, `prompt_version`, `filename`, and error rows add `error`/`attempts`).